# MSE 465/665 · MC Lab B: Same problem, production code

In Lab A you wrote a Monte Carlo neutron code yourself. In this lab you run **OpenMC**, a production Monte Carlo code used to design reactors and shields. First you give it Lab A's water slab, so you can check it against your own code. Then you give it the problem S10 left open: how many scattered photons get through a lead shield, and how much dose they deliver.

| Part | What you do | Checks against |
|---|---|---|
| 0 | Install OpenMC and set your ID | a test run against $e^{-\Sigma_t x}$ |
| 1 | Rerun Lab A's water slab in OpenMC | $e^{-\Sigma_t x}$ and your own Lab A result |
| 2 | Send Cs-137 photons through lead and through water | the half-thicknesses from S10 (S&F Table C.3) |
| 3 | Find the dose rate behind the shield | S12-A's dose rate (S&F Eq. 9.6) |

**Python needed:** the same as Lab A. You never write OpenMC code: each OpenMC run is one call to a helper function, and each blank you fill in is one line of physics.

**MSE 665 students** also complete the cells marked **[665]**, where you build an OpenMC model yourself. MSE 465 students skip those cells.

**Plan on about 3 hours.** The checkpoint is Part 0: a screenshot of the PASS line from the setup check.

## Before you start

1. **Save your own copy:** *File > Save a copy in Drive*. Work only in that copy.
2. **Run cells from top to bottom** (Shift + Enter). Every cell uses results from the cells above it.
3. **Every check cell prints PASS or NOT YET.** NOT YET always says what to fix.
4. **Colab forgets OpenMC when a session ends** (after about 90 minutes without activity, or when you close the tab). When you come back, run *Runtime > Run all*. The install cell takes about 30 seconds, and every result repeats exactly because it is seeded from your ID.

### When something goes wrong

| What you see | What it means | Fix |
|---|---|---|
| `NOT YET ... still blank` | A line still contains `___` | Replace `___` with your expression and run the cell |
| `NameError: name '...' is not defined` | A cell above this one was not run | *Runtime > Run before*, then run this cell |
| `NOT YET  OpenMC is not installed` or `FileNotFoundError: ... 'openmc'` | The session restarted and OpenMC is gone | *Runtime > Run all* |
| The install cell prints `NOT YET  OpenMC did not install` | The download or the build failed | *Runtime > Disconnect and delete runtime*, then *Runtime > Run all*. If it fails again, post the lines it printed on the discussion board |
| `TypeError: unsupported operand type(s) for ^` | You wrote `^` for a power | Python writes σ² as `sigma**2` |
| `SyntaxError` | A typo: an unmatched parenthesis or a missing quote | Compare your line with the comment beside it |

### What you submit

1. **Checkpoint:** a screenshot of the PASS line from the setup check (Part 0.4).
2. **Final:** the answer sheet, filled in from the values the notebook prints, and this notebook (*File > Download > Download .ipynb*).

---
## Part 0 · Install OpenMC

**0.1 Your ID.** As in Lab A, every OpenMC run in this lab is seeded from your ID, so your results are your own and repeat exactly when you rerun a cell. Type the **last four digits of your student ID** between the quotes and run the cell.

In [ ]:
MY_ID = "0000"   # last four digits of your student ID, inside the quotes

**0.2 Install.** Run the next cell. The install cell installs OpenMC (free, open-source software) and about 9 MB of nuclear data, then prints **DONE**. The install takes about 30 seconds.

In [ ]:
# @title Install OpenMC: run this cell once per session (about 30 s; up to 4 min if it must build OpenMC)
# MSE 465/665 MC Lab B. Installs OpenMC 0.16.0 and the lab's nuclear data. You do not need to read this code.
import os, sys, json, time, shutil, subprocess, urllib.request, importlib

LAB_FILES = "https://raw.githubusercontent.com/JMoonUNR/MSE465-labs/main/labB"            # course repository folder with the OpenMC build and the data files
LAB_FILES = os.environ.get("LABB_FILES_URL", LAB_FILES)
DATA = os.environ.setdefault("LABB_DATA", os.path.expanduser("~/labB_data"))
TAG = "0.16.0"
DATA_FILES = ["photon/Pb.h5", "photon/H.h5", "photon/O.h5", "neutron/Pb208.h5", "neutron/H1.h5", "neutron/O16.h5"]
APT = {"jammy": "libhdf5-103-1 libhdf5-hl-100", "noble": "libhdf5-103-1t64 libhdf5-hl-100t64"}


def _sh(cmd, timeout=900):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    return r.returncode == 0, (r.stdout + r.stderr)[-2000:]


def _apt(pkgs):
    ok, out = _sh("apt-get install -y -q " + pkgs)
    if not ok:
        _sh("apt-get update -q")
        ok, out = _sh("apt-get install -y -q " + pkgs)
    return ok, out


def _openmc_ready():
    if shutil.which("openmc") is None:
        return False
    importlib.invalidate_caches()
    try:
        import openmc  # noqa: F401
        return True
    except Exception:
        return False


def _fetch(url, dest):
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    urllib.request.urlretrieve(url, dest + ".part")
    os.replace(dest + ".part", dest)


def _prebuilt(log):
    try:
        rel = dict(l.strip().split("=", 1) for l in open("/etc/os-release") if "=" in l)
    except OSError:
        log.append("no /etc/os-release")
        return False
    code = rel.get("VERSION_CODENAME", "").strip('"')
    if code not in APT:
        log.append("no prebuilt OpenMC for " + (code or "this system"))
        return False
    ok, out = _apt(APT[code])
    if not ok:
        log.append("apt: " + out)
        return False
    tarball = "/tmp/openmc-%s-%s-x86_64.tar.gz" % (TAG, code)
    wheel = "/tmp/openmc-%s-py3-none-any.whl" % TAG
    try:
        _fetch("%s/openmc/%s" % (LAB_FILES, os.path.basename(tarball)), tarball)
        _fetch("%s/openmc/%s" % (LAB_FILES, os.path.basename(wheel)), wheel)
    except Exception as ex:
        log.append("download: %s" % ex)
        return False
    ok, out = _sh("tar -xzf %s -C /opt" % tarball)
    if not ok:
        log.append("tar: " + out)
        return False
    os.environ["PATH"] = "/opt/openmc/bin:" + os.environ["PATH"]
    ok, out = _sh("%s -m pip install -q %s" % (sys.executable, wheel))
    if not ok:
        log.append("pip: " + out)
        return False
    ok, out = _sh("openmc --version")
    if not ok:
        log.append("openmc --version: " + out)
        return False
    return _openmc_ready()


def _from_source(log):
    print("No prebuilt OpenMC fits this machine. Building OpenMC from source: about 3 minutes, please wait.", flush=True)
    steps = [
        "apt-get install -y -q libhdf5-dev git cmake g++",
        "rm -rf /tmp/openmc-src && git clone -q --depth 1 --branch v%s https://github.com/openmc-dev/openmc.git /tmp/openmc-src" % TAG,
        "cd /tmp/openmc-src && mkdir -p build && cd build && cmake .. -DCMAKE_BUILD_TYPE=Release -DOPENMC_BUILD_TESTS=off -DCMAKE_INSTALL_PREFIX=/usr/local",
        "cd /tmp/openmc-src/build && make -j2 install",
        "%s -m pip install -q --no-deps /tmp/openmc-src && %s -m pip install -q endf uncertainties" % (sys.executable, sys.executable),
    ]
    for cmd in steps:
        ok, out = _apt(cmd[len("apt-get install -y -q "):]) if cmd.startswith("apt-get") else _sh(cmd, timeout=1800)
        if not ok:
            log.append("%s: %s" % (cmd.split("&&")[-1].strip()[:60], out))
            return False
    return _openmc_ready()


def install():
    t0 = time.time()
    log = []
    route = None
    if _openmc_ready():
        route = "already installed"
    elif "__LAB_FILES" in LAB_FILES:
        log.append("the course files address (LAB_FILES) is not set in this notebook")
    elif _prebuilt(log):
        route = "prebuilt"
    if route is None and "__LAB_FILES" not in LAB_FILES and _from_source(log):
        route = "built from source"
    if route is None:
        print("NOT YET   OpenMC did not install. Try: Runtime > Disconnect and delete runtime, then Runtime > Run all.")
        print("          If this message comes back, post the lines below on the discussion board.")
        for line in log:
            print("          " + line.strip().replace("\n", "\n          ")[-600:])
        return
    try:
        for f in DATA_FILES:
            dest = os.path.join(DATA, f)
            if not os.path.exists(dest):
                _fetch("%s/data/%s" % (LAB_FILES, f), dest)
    except Exception as ex:
        print("NOT YET   The nuclear data files did not download (%s). Run this cell again." % ex)
        return
    import openmc
    lib = openmc.data.DataLibrary()
    for f in DATA_FILES:
        lib.register_file(os.path.join(DATA, f))
    lib.export_to_xml(os.path.join(DATA, "cross_sections.xml"))
    secs = time.time() - t0
    if route != "already installed":
        json.dump({"route": route, "seconds": round(secs)}, open(os.path.join(DATA, "install.json"), "w"))
    mb = sum(os.path.getsize(os.path.join(DATA, f)) for f in DATA_FILES) / 1e6
    print("DONE      OpenMC %s ready (%s, %d s). Nuclear data: %.1f MB in %s." % (openmc.__version__, route, secs, mb, DATA))


install()


**0.3 Helper code.** Run the next cell. The helper cell loads the lab's helper functions (hidden; you do not need to read them).

In [ ]:
# @title Helper code: run this cell once. You do not need to read it.
# MSE 465/665 MC Lab B helper code: OpenMC runs, checks and plots.
import math, os, json, time, shutil, tempfile, warnings
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
import openmc, openmc.data

BLUE, BLUE_L, MARI, INK, GRAY, RED = "#156082", "#9DBFD0", "#D69A1A", "#3A362D", "#9A9382", "#B3261E"
plt.rcParams.update({"figure.figsize": (6.4, 4.2), "font.size": 11, "axes.edgecolor": GRAY,
                     "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
                     "axes.spines.top": False, "axes.spines.right": False})

LAB = {"water1g": {"Sigma_t": 2.1963, "Sigma_s": 2.174, "Sigma_a": 0.0223}, "labA": {"Ttot2": 0.2522, "R10": 0.75, "N": 1000000, "keyT": 0.2559, "keyS": 0.0031}, "E_MeV": 0.662, "s10a": {"muRho": 0.1043, "rho": 11.35, "mu": 1.184, "xHalf": 0.585, "page": 595}, "s10water": {"muRho": 0.08554, "xHalf": 8.1, "rho": 1.0, "page": 592}, "src": {"A_Ci": 10.0, "r_cm": 200.0, "yield_": 0.849, "S": 314100000000.0, "phi0": 624900.0, "tPb": 2.0, "tC": 10.0, "phi": 9760.0}, "s12a": {"K": 1.602e-10, "muenW": 0.03257, "DuGyh": 121}, "compton": {"mec2": 0.511, "E90_keV": 288, "E180_keV": 184}, "muenW_rows": [[0.001, 4065.0000000000005], [0.0015, 1372.0], [0.002, 615.2], [0.003, 191.70000000000002], [0.004, 81.91000000000001], [0.005, 41.879999999999995], [0.006, 24.049999999999997], [0.008, 9.915], [0.01, 4.944], [0.015, 1.374], [0.02, 0.5503], [0.03, 0.1557], [0.04, 0.06947], [0.05, 0.04223], [0.06, 0.0319], [0.08, 0.02597], [0.1, 0.02546], [0.15, 0.027639999999999998], [0.2, 0.029670000000000002], [0.3, 0.031920000000000004], [0.4, 0.03279], [0.5, 0.03299], [0.6, 0.03284], [0.8, 0.03206], [1.0, 0.031030000000000002]]}
WATER = LAB["water1g"]                 # one-group water, Lab A and S9-A
E0 = LAB["E_MeV"]                      # Cs-137 photon energy, MeV
E0_eV = E0 * 1e6
DATA = os.environ.get("LABB_DATA", os.path.expanduser("~/labB_data"))
XS = os.path.join(DATA, "cross_sections.xml")
DELTA = 0.01                            # cm, thin void layer behind the shield where the fluence is scored


class _Blank:
    """Stands in for ___ so an unfinished line runs without crashing."""
    def __repr__(self):
        return "___ (still blank)"
    __str__ = __repr__

    def __bool__(self):
        return False

    def _b(self, *a, **k):
        return BLANK
    __add__ = __radd__ = __sub__ = __rsub__ = __mul__ = __rmul__ = _b
    __truediv__ = __rtruediv__ = __pow__ = __rpow__ = __neg__ = __abs__ = _b
    __lt__ = __le__ = __gt__ = __ge__ = __and__ = __rand__ = __or__ = __ror__ = __invert__ = _b
    __hash__ = object.__hash__

    def __array_ufunc__(self, *a, **k):
        return BLANK

    def __getattr__(self, name):
        if name.startswith("_"):
            raise AttributeError(name)
        return self._b


BLANK = _Blank()
___ = BLANK
ANSWERS, _ORDER, _STATE = {}, {}, {}


def _blank(v):
    if isinstance(v, _Blank):
        return True
    if isinstance(v, (list, tuple)):
        return any(_blank(i) for i in v)
    return False


def _ok(msg):
    print("PASS      " + msg)


def _no(msg):
    print("NOT YET   " + msg)


def _record(key, text):
    if key not in _ORDER:
        _ORDER[key] = len(_ORDER)
    ANSWERS[key] = text


def _id_digits():
    raw = str(globals().get("MY_ID", "")).strip()
    return "".join(ch for ch in raw if ch.isdigit())


def _seed(part, k=0):
    """OpenMC seed for one run, from MY_ID, so reruns repeat exactly and every student differs."""
    d = _id_digits()
    base = int(d) if d else 0
    return 1 + (465 * 1_000_003 + base * 10_007 + part * 101 + k) % (2**62)


def _num(v):
    try:
        a = np.asarray(v, dtype=float)
    except Exception:
        return None
    return a


# ------------------------------------------------------------------ materials and models
def _lead():
    m = openmc.Material(name="lead")
    # Photon physics depends only on the element, so one lead isotope stands in for all four,
    # at the atom density of natural lead. This keeps the data download small.
    n = LAB["s10a"]["rho"] * openmc.data.AVOGADRO / openmc.data.atomic_weight("Pb") * 1e-24
    m.add_nuclide("Pb208", 1.0)
    m.set_density("atom/b-cm", n)
    return m


def _water():
    m = openmc.Material(name="water")
    m.add_nuclide("H1", 2.0)
    m.add_nuclide("O16", 1.0)
    m.set_density("g/cm3", 1.0)
    return m


RHO = {"lead": LAB["s10a"]["rho"], "water": 1.0}


def _mg_library(path, scattering):
    g = openmc.mgxs.EnergyGroups([0.0, 20.0e6])
    d = openmc.XSdata("water_1g", g)
    d.order = 0
    s = WATER["Sigma_s"] if scattering else 0.0
    d.set_total([WATER["Sigma_t"]])
    d.set_absorption([WATER["Sigma_t"] - s])
    d.set_scatter_matrix(np.array([[[s]]]))
    lib = openmc.MGXSLibrary(g)
    lib.add_xsdata(d)
    lib.export_to_hdf5(path)


def _sides():
    return (+openmc.YPlane(-1, boundary_type="reflective") & -openmc.YPlane(1, boundary_type="reflective")
            & +openmc.ZPlane(-1, boundary_type="reflective") & -openmc.ZPlane(1, boundary_type="reflective"))


def _spectrum_bins():
    return np.r_[np.arange(0.0, 660e3 + 1, 5e3), E0_eV - 1, E0_eV + 1, 700e3]


def _dose_function():
    """Eq. 9.6 per photon: D/Phi = 1.602e-10 E (mu_en/rho) in Gy cm2, with water mu_en/rho from Table C.3 (p. 592)."""
    rows = np.array(LAB["muenW_rows"])
    E = rows[:, 0]
    return E * 1e6, LAB["s12a"]["K"] * E * rows[:, 1]


def _model(material, x, particle, n, seed, scattering=True):
    openmc.reset_auto_ids()
    wd = tempfile.mkdtemp(prefix="labB_")
    s = openmc.Settings()
    s.run_mode = "fixed source"
    s.batches = 20
    s.particles = n // 20
    s.seed = seed
    s.output = {"tallies": False}
    sides = _sides()
    front = openmc.XPlane(0.0, boundary_type="vacuum", name="front face")
    tallies = []
    if particle == "neutron":
        lib = os.path.join(wd, "mgxs.h5")
        _mg_library(lib, scattering)
        mat = openmc.Material(name="water, one group")
        mat.set_density("macro", 1.0)
        mat.add_macroscopic("water_1g")
        mats = openmc.Materials([mat])
        mats.cross_sections = lib
        back = openmc.XPlane(x, boundary_type="vacuum", name="back face")
        cells = [openmc.Cell(fill=mat, region=+front & -back & sides, name="slab")]
        s.energy_mode = "multi-group"
        s.source = openmc.IndependentSource(space=openmc.stats.Point((1e-9, 0, 0)),
                                            angle=openmc.stats.Monodirectional((1, 0, 0)),
                                            energy=openmc.stats.Discrete([1.0e6], [1.0]))
        t = openmc.Tally(name="transmitted"); t.filters = [openmc.SurfaceFilter(back)]; t.scores = ["current"]
        r = openmc.Tally(name="reflected"); r.filters = [openmc.SurfaceFilter(front)]; r.scores = ["current"]
        tallies = [t, r]
    else:
        mat = _lead() if material == "lead" else _water()
        mats = openmc.Materials([mat])
        mats.cross_sections = XS
        back = openmc.XPlane(x, name="back face")
        end = openmc.XPlane(x + DELTA, boundary_type="vacuum", name="end of fluence layer")
        cells = [openmc.Cell(fill=mat, region=+front & -back & sides, name="shield"),
                 openmc.Cell(fill=None, region=+back & -end & sides, name="fluence layer")]
        s.photon_transport = True
        s.source = openmc.IndependentSource(space=openmc.stats.Point((1e-9, 0, 0)),
                                            angle=openmc.stats.Monodirectional((1, 0, 0)),
                                            energy=openmc.stats.Discrete([E0_eV], [1.0]), particle="photon")
        pf = openmc.ParticleFilter(["photon"])
        layer = openmc.CellFilter(cells[1])
        full = openmc.EnergyFilter([E0_eV - 1, E0_eV + 1])
        t1 = openmc.Tally(name="narrow"); t1.filters = [openmc.SurfaceFilter(back), pf, full]; t1.scores = ["current"]
        t2 = openmc.Tally(name="fluence"); t2.filters = [layer, pf]; t2.scores = ["flux"]
        t3 = openmc.Tally(name="fluence narrow"); t3.filters = [layer, pf, full]; t3.scores = ["flux"]
        t4 = openmc.Tally(name="spectrum"); t4.filters = [layer, pf, openmc.EnergyFilter(_spectrum_bins())]; t4.scores = ["flux"]
        Ed, fd = _dose_function()
        dfil = openmc.EnergyFunctionFilter(Ed, fd)
        dfil.interpolation = "log-log"
        t5 = openmc.Tally(name="dose"); t5.filters = [layer, pf, dfil]; t5.scores = ["flux"]
        t6 = openmc.Tally(name="strict"); t6.filters = [openmc.SurfaceFilter(back), pf, full,
                                                         openmc.MuSurfaceFilter([0.999999, 1.0])]; t6.scores = ["current"]
        tallies = [t1, t2, t3, t4, t5, t6]
    model = openmc.Model(openmc.Geometry(cells), mats, s, openmc.Tallies(tallies))
    return model, wd


def _run(model, wd):
    cwd = os.getcwd()
    os.chdir(wd)
    try:
        t0 = time.time()
        sp = model.run(output=False)
        dt = time.time() - t0
        out = {}
        with openmc.StatePoint(sp) as st:
            for t in st.tallies.values():
                out[t.name] = (t.mean.ravel().copy(), t.std_dev.ravel().copy())
    finally:
        os.chdir(cwd)
    _STATE["last_model_dir"] = wd
    return out, dt


def _ready(particle):
    if shutil.which("openmc") is None:
        _no("OpenMC is not installed in this session. Run Runtime > Run all; the install cell takes about 30 s.")
        return False
    if particle == "photon" and not os.path.exists(XS):
        _no("The nuclear data files are missing. Run the install cell (Part 0.2) again, then this cell.")
        return False
    return True


def run_sweep(material, thicknesses, particle, energy_MeV=None, n_particles=100_000, scattering=True, part=0):
    """Run OpenMC once per slab thickness. Returns a dictionary of results, one value per thickness."""
    if _blank(thicknesses):
        _no("The thickness list is still blank.")
        return BLANK
    if not _ready(particle):
        return BLANK
    xs = np.array(thicknesses, dtype=float)
    if particle == "photon" and energy_MeV is not None and abs(energy_MeV - E0) > 1e-9:
        print("Note: this lab's photon runs use 0.662 MeV, the Cs-137 line; energy_MeV is ignored.")
    res = {"material": material, "particle": particle, "x": xs, "N": n_particles, "scattering": scattering}
    keys = ["T", "sT", "R", "sR"] if particle == "neutron" else \
        ["T_narrow", "sT_narrow", "Phi_total", "sPhi_total", "Phi_narrow", "sPhi_narrow", "D", "sD", "T_strict", "sT_strict"]
    for k in keys:
        res[k] = np.zeros(xs.size)
    spectra = []
    t_all = 0.0
    for i, x in enumerate(xs):
        model, wd = _model(material, x, particle, n_particles, _seed(part, i), scattering)
        out, dt = _run(model, wd)
        t_all += dt
        if particle == "neutron":
            res["T"][i], res["sT"][i] = abs(out["transmitted"][0][0]), out["transmitted"][1][0]
            res["R"][i], res["sR"][i] = abs(out["reflected"][0][0]), out["reflected"][1][0]
        else:
            res["T_narrow"][i], res["sT_narrow"][i] = out["narrow"][0][0], out["narrow"][1][0]
            res["Phi_total"][i], res["sPhi_total"][i] = out["fluence"][0][0] / DELTA, out["fluence"][1][0] / DELTA
            res["Phi_narrow"][i], res["sPhi_narrow"][i] = out["fluence narrow"][0][0] / DELTA, out["fluence narrow"][1][0] / DELTA
            res["D"][i], res["sD"][i] = out["dose"][0][0] / DELTA, out["dose"][1][0] / DELTA
            res["T_strict"][i], res["sT_strict"][i] = out["strict"][0][0], out["strict"][1][0]
            spectra.append(out["spectrum"][0] / DELTA)
    if particle == "photon":
        res["spectra"] = spectra
    res["seconds"] = t_all
    _print_sweep(res)
    return res


def _print_sweep(r):
    print("%s, %s, %s particles per thickness, %.1f s of OpenMC time" % (
        r["material"] + ("" if r["scattering"] else " (no scattering)"), r["particle"] + "s", f"{r['N']:,}", r["seconds"]))
    if r["particle"] == "neutron":
        print("  x (cm)    T (transmitted) +/- sigma      R (reflected) +/- sigma")
        for i, x in enumerate(r["x"]):
            print("  %5.2f     %.5f +/- %.5f            %.5f +/- %.5f" % (x, r["T"][i], r["sT"][i], r["R"][i], r["sR"][i]))
    else:
        print("  x (cm)    T_narrow +/- sigma        Phi_total +/- sigma      Phi_narrow")
        for i, x in enumerate(r["x"]):
            print("  %5.2f     %.5f +/- %.5f      %.5f +/- %.5f      %.5f" % (
                x, r["T_narrow"][i], r["sT_narrow"][i], r["Phi_total"][i], r["sPhi_total"][i], r["Phi_narrow"][i]))
        print("  T_narrow: share of photons leaving the back face at the full 662 keV.")
        print("  Phi: photon fluence just behind the shield, per photon per cm2 sent in (1 means no shield).")


def at(run, x):
    """The transmission and its sigma at one thickness of a neutron sweep."""
    if not isinstance(run, dict):
        _no("Run the sweep cell above first.")
        return BLANK, BLANK
    i = int(np.argmin(abs(run["x"] - x)))
    if abs(run["x"][i] - x) > 1e-9:
        _no("Thickness %g cm is not in this sweep." % x)
        return BLANK, BLANK
    print("x = %g cm: T = %.5f +/- %.5f" % (x, run["T"][i], run["sT"][i]))
    return float(run["T"][i]), float(run["sT"][i])


def show_model():
    """Print the OpenMC input of the most recent run."""
    wd = _STATE.get("last_model_dir")
    if not wd:
        _no("Run OpenMC first.")
        return
    p = os.path.join(wd, "model.xml")
    files = [p] if os.path.exists(p) else [os.path.join(wd, f) for f in ("materials.xml", "geometry.xml", "settings.xml", "tallies.xml")]
    for f in files:
        if os.path.exists(f):
            txt = open(f).read().replace("<?xml version='1.0' encoding='utf-8'?>\n", "").replace(wd + "/", "")
            print("=" * 20, os.path.basename(f), "=" * 20)
            print(txt.strip())
    lib = os.path.join(wd, "mgxs.h5")
    if os.path.exists(lib):
        import h5py
        with h5py.File(lib, "r") as fh:
            g = fh["water_1g"]
            key = [k for k in g.keys() if k.endswith("K")][0]
            d = g[key]
            print("=" * 20, "mgxs.h5 (one energy group, per cm)", "=" * 20)
            print("total %.4f   absorption %.4f   scatter %.4f" % (
                float(np.ravel(d["total"][()])[0]), float(np.ravel(d["absorption"][()])[0]),
                float(np.ravel(d["scatter_data"]["scatter_matrix"][()])[0])))


# ------------------------------------------------------------------ setup
def check_setup():
    d = _id_digits()
    if len(d) != 4 or d == "0000":
        _no('Set MY_ID to the last four digits of your student ID, inside the quotes '
            '(for example MY_ID = "0472"). Run that cell, then run this one again.')
        return
    if shutil.which("openmc") is None:
        _no("OpenMC is not installed. Run the install cell above; it prints DONE when OpenMC is ready.")
        return
    if not os.path.exists(XS):
        _no("The nuclear data files are missing. Run the install cell above again.")
        return
    model, wd = _model("water", 1.0, "neutron", 20_000, _seed(0), scattering=False)
    out, dt = _run(model, wd)
    T, sT = abs(out["transmitted"][0][0]), out["transmitted"][1][0]
    want = math.exp(-WATER["Sigma_t"] * 1.0)
    if abs(T - want) > 4 * sT:
        _no("The OpenMC test run gave T = %.4f, expected %.4f. Rerun the install cell, then this cell." % (T, want))
        return
    info = {}
    try:
        info = json.load(open(os.path.join(DATA, "install.json")))
    except Exception:
        pass
    _STATE["setup"] = True
    route = info.get("route", "already installed")
    secs = info.get("seconds")
    _ok("Setup complete. MY_ID = %s. OpenMC %s (%s%s). Test run: T(1 cm) = %.4f +/- %.4f against %.4f, in %.1f s." % (
        d, openmc.__version__, route, ", %d s to install" % secs if secs else "", T, sT, want, dt))
    _record("0. MY_ID", d)
    _record("0a. OpenMC version and install", "%s, %s%s" % (openmc.__version__, route, ", %d s" % secs if secs else ""))


# ------------------------------------------------------------------ Part 1
def check_part1a(T_exact, res0):
    _STATE["p1a"] = False
    if res0 is BLANK or not isinstance(res0, dict):
        _no("Run the pure-absorber sweep cell first.")
        return
    if _blank(T_exact):
        _no("The line T_exact = ___ is still blank.")
        return
    a = _num(T_exact)
    x = res0["x"]
    want = np.exp(-WATER["Sigma_t"] * x)
    if a is None or a.shape != x.shape:
        _no("T_exact should hold one value per thickness (%d of them). Use the array x, not one number." % x.size)
        return
    if np.any(a > 1):
        _no("T_exact is above 1 at some thickness. A transmission cannot exceed 1: check the minus sign in the exponent.")
        return
    if np.allclose(a, np.exp(-WATER["Sigma_s"] * x), rtol=1e-6) or np.allclose(a, np.exp(-WATER["Sigma_a"] * x), rtol=1e-6):
        _no("T_exact uses the wrong cross section. Every collision removes a neutron from the uncollided beam: use Sigma_t.")
        return
    if not np.allclose(a, want, rtol=1e-6):
        _no("T_exact should be exp(-Sigma_t x) at each thickness. Numpy writes e to a power as np.exp(...).")
        return
    z = (res0["T"] - want) / res0["sT"]
    _STATE["p1a"] = True
    print("  x (cm)   OpenMC T +/- sigma       exp(-Sigma_t x)   difference (sigma)")
    for i in range(x.size):
        print("  %5.2f    %.5f +/- %.5f      %.5f          %+.2f" % (x[i], res0["T"][i], res0["sT"][i], want[i], z[i]))
    worst = float(np.max(np.abs(z)))
    if worst > 4:
        _no("One point sits %.1f sigma from the formula. Check that the sweep cell ran with scattering=False." % worst)
        return
    _ok("With scattering turned off, OpenMC reproduces exp(-Sigma_t x) at every thickness.")
    i2 = int(np.argmin(abs(x - 2.0)))
    _record("1a. no scattering, x = 2 cm: T", "%.5f +/- %.5f (exp: %.5f, %+.2f sigma)" % (res0["T"][i2], res0["sT"][i2], want[i2], z[i2]))


def check_part1b(z, T_openmc, sigma_openmc, T_labA, sigma_labA, res1):
    _STATE["p1b"] = False
    if res1 is BLANK or not isinstance(res1, dict):
        _no("Run the scattering sweep cell first.")
        return
    if _blank([T_labA, sigma_labA]):
        _no("Copy T_total at 2 cm and its sigma from your Lab A answer sheet (item 4a) into T_labA and sigma_labA, "
            "then run that cell again. No Lab A result? Use T_labA = %.4f and sigma_labA = %.4f." % (LAB["labA"]["keyT"], LAB["labA"]["keyS"]))
        return
    if _blank(z):
        _no("The line z = ___ is still blank.")
        return
    try:
        zv = float(z)
    except Exception:
        _no("z should be one number.")
        return
    want = (T_openmc - T_labA) / math.sqrt(sigma_openmc ** 2 + sigma_labA ** 2)
    if abs(zv - want) > 1e-6 * max(1, abs(want)):
        if abs(zv - (T_openmc - T_labA) / (sigma_openmc + sigma_labA)) < 1e-6 * max(1, abs(want)):
            _no("Uncertainties add in quadrature, not linearly: divide by np.sqrt(sigma_openmc**2 + sigma_labA**2).")
        elif abs(zv - (T_openmc - T_labA) / (sigma_openmc ** 2 + sigma_labA ** 2)) < 1e-6 * max(1, abs(want)):
            _no("You divided by the variance. Take the square root: np.sqrt(sigma_openmc**2 + sigma_labA**2).")
        elif abs(zv + want) < 1e-6 * max(1, abs(want)):
            _no("The sign is flipped. Write OpenMC minus Lab A, so z > 0 means OpenMC transmits more.")
        else:
            _no("z should be (T_openmc - T_labA) / sqrt(sigma_openmc**2 + sigma_labA**2). Check the parentheses.")
        return
    _STATE["p1b"] = True
    _ok("OpenMC T_total(2 cm) = %.4f +/- %.4f;  your Lab A = %.4f +/- %.4f;  difference = %+.2f sigma"
        % (T_openmc, sigma_openmc, T_labA, sigma_labA, zv))
    zr = (T_openmc - LAB["labA"]["Ttot2"]) / math.sqrt(sigma_openmc ** 2 + LAB["labA"]["Ttot2"] * (1 - LAB["labA"]["Ttot2"]) / LAB["labA"]["N"])
    print("          Lab A's 10^6-history reference is %.4f: OpenMC sits %+.2f sigma from it." % (LAB["labA"]["Ttot2"], zr))
    _record("1b. x = 2 cm: OpenMC T_total", "%.4f +/- %.4f" % (T_openmc, sigma_openmc))
    _record("1b. z against your Lab A result", "%+.2f" % zv)
    i10 = int(np.argmin(abs(res1["x"] - 10.0)))
    _record("1c. x = 10 cm: R", "%.4f +/- %.4f" % (res1["R"][i10], res1["sR"][i10]))
    fig, ax = plt.subplots()
    x = res1["x"]
    xx = np.linspace(0, x.max(), 200)
    ax.semilogy(xx, np.exp(-WATER["Sigma_t"] * xx), color=BLUE, lw=1.8, label=r"uncollided, $e^{-\Sigma_t x}$")
    ax.errorbar(x, res1["T"], yerr=res1["sT"], fmt="o-", color=MARI, ms=5, capsize=3, label="OpenMC, total transmitted")
    ax.plot(x, res1["R"], "s--", color=GRAY, ms=5, label="OpenMC, reflected")
    ax.errorbar([2.0], [T_labA], yerr=[sigma_labA], fmt="D", color=INK, ms=7, capsize=3, label="your Lab A, 2 cm")
    ax.set_ylim(1e-6, 1.5)
    ax.set_xlabel("water slab thickness x (cm)")
    ax.set_ylabel("fraction of incident neutrons")
    ax.legend(frameon=False, loc="lower left")
    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------------ Part 2
def library_mu(material):
    """Narrow-beam mu at 662 keV from the photon data files OpenMC reads (no coherent scattering), 1/cm."""
    m = _lead() if material == "lead" else _water()
    dens = m.get_nuclide_atom_densities()          # atoms per barn-cm
    elems = {"Pb208": "Pb", "H1": "H", "O16": "O"}
    mu = 0.0
    for nuc, n in dens.items():
        ph = openmc.data.IncidentPhoton.from_hdf5(os.path.join(DATA, "photon", elems[nuc] + ".h5"))
        sig = float(ph.reactions[504].xs(E0_eV))
        sig += sum(float(r.xs(E0_eV)) for mt, r in ph.reactions.items() if 534 <= mt <= 572)
        sig += sum(float(ph.reactions[mt].xs(E0_eV)) for mt in (515, 517) if mt in ph.reactions)
        mu += n * sig
    return mu


def _weighted_mu(run):
    T, sT, x = run["T_narrow"], run["sT_narrow"], run["x"]
    mu_i = -np.log(T) / x
    s_i = sT / (T * x)
    w = 1 / s_i ** 2
    return float(np.sum(w * mu_i) / np.sum(w)), float(1 / math.sqrt(np.sum(w)))


def check_mu(run, mu):
    """Check your narrow-beam mu, one value per thickness. Returns their weighted mean."""
    if run is BLANK or not isinstance(run, dict):
        _no("Run the sweep cell first.")
        return BLANK
    if _blank(mu):
        _no("The line mu = ___ is still blank.")
        return BLANK
    a = _num(mu)
    T, x = run["T_narrow"], run["x"]
    want = -np.log(T) / x
    if a is None or a.shape != x.shape:
        _no("mu should hold one value per thickness (%d of them). Use the arrays T_narrow and x." % x.size)
        return BLANK
    if np.allclose(a, -want, rtol=1e-6):
        _no("mu is negative. ln of a transmission below 1 is negative, so mu = -ln(T_narrow) / x.")
        return BLANK
    if np.allclose(a, -np.log10(T) / x, rtol=1e-6):
        _no("np.log10 is the base-10 log. The attenuation law uses the natural log: np.log.")
        return BLANK
    if np.allclose(a, -np.log(T) * x, rtol=1e-6):
        _no("Divide by the thickness x; do not multiply.")
        return BLANK
    if not np.allclose(a, want, rtol=1e-6):
        _no("Rearrange T_narrow = exp(-mu x) for mu, one value per thickness.")
        return BLANK
    m, sm = _weighted_mu(run)
    lib = library_mu(run["material"])
    z = (m - lib) / sm
    run["mu"], run["s_mu"], run["mu_lib"] = m, sm, lib
    for i in range(x.size):
        print("  x = %5.2f cm   mu = %.4f per cm" % (x[i], a[i]))
    if abs(z) > 4:
        _no("Weighted mean mu = %.4f +/- %.4f per cm sits %.1f sigma from the data library's %.4f. Check that the sweep cell ran for this material." % (m, sm, z, lib))
        return BLANK
    _ok("%s: mu = %.4f +/- %.4f per cm (weighted mean over the thicknesses); the photon data library gives %.4f (%+.2f sigma)."
        % (run["material"], m, sm, lib, z))
    _STATE["mu_" + run["material"]] = (m, sm, lib)
    _record(("2a. mu, lead (per cm)" if run["material"] == "lead" else "2g. mu, water (per cm)"),
            "%.4f +/- %.4f (library %.4f)" % (m, sm, lib))
    return m


def check_xhalf(x_half, mu_lead):
    _STATE["p2xh"] = False
    if "mu_lead" not in _STATE:
        _no("Get PASS on mu for lead first.")
        return
    if _blank(x_half):
        _no("The line x_half = ___ is still blank.")
        return
    m, sm, lib = _STATE["mu_lead"]
    want = math.log(2) / m
    try:
        xv = float(x_half)
    except Exception:
        _no("x_half should be one number. Use mu_lead, the weighted mean, not the array mu.")
        return
    if abs(xv - math.log10(2) / m) < 1e-9:
        _no("np.log10 is the base-10 log. The half-thickness uses ln 2: np.log(2).")
        return
    if abs(xv - 1 / m) < 1e-9:
        _no("1/mu is the mean free path. The half-thickness is ln 2 / mu.")
        return
    if abs(xv - want) > 1e-9 * want:
        _no("x_half should be np.log(2) / mu_lead.")
        return
    sx = want * sm / m
    xl = math.log(2) / lib
    s10 = LAB["s10a"]["xHalf"]
    _STATE["p2xh"] = True
    _ok("Half-thickness of lead at 662 keV: %.4f +/- %.4f cm. Data library: %.4f cm (%+.2f sigma). S10-A from Table C.3: %.3f cm (%+.1f%%)."
        % (want, sx, xl, (want - xl) / sx, s10, 100 * (want / s10 - 1)))
    _record("2a. x_half, lead (cm)", "%.4f +/- %.4f (library %.4f; S10-A %.3f, %+.1f%%)" % (want, sx, xl, s10, 100 * (want / s10 - 1)))


def check_B(B, run):
    _STATE["p2B"] = False
    if run is BLANK or not isinstance(run, dict):
        _no("Run the lead sweep first.")
        return
    if _blank(B):
        _no("The line B = ___ is still blank.")
        return
    want = run["Phi_total"] / run["Phi_narrow"]
    b = _num(B)
    if b is None or b.shape != want.shape:
        _no("B should hold one value per thickness (%d of them). Use the arrays Phi_total and Phi_narrow." % want.size)
        return
    if np.allclose(b, 1 / want, rtol=1e-9):
        _no("B is upside down. The buildup factor is total fluence / narrow-beam fluence, so B >= 1.")
        return
    if np.allclose(b, run["Phi_total"] / run["T_narrow"], rtol=1e-9) and not np.allclose(b, want, rtol=1e-3):
        _no("Divide fluence by fluence: use Phi_narrow, not T_narrow.")
        return
    if not np.allclose(b, want, rtol=1e-9):
        _no("B should be Phi_total / Phi_narrow, one value per thickness.")
        return
    sB = want * np.sqrt((run["sPhi_total"] / run["Phi_total"]) ** 2 + (run["sPhi_narrow"] / run["Phi_narrow"]) ** 2)
    run["B"], run["sB"] = want, sB
    for x, bb, ss in zip(run["x"], want, sB):
        print("  x = %4.1f cm   B = %.3f +/- %.3f" % (x, bb, ss))
    if not (np.all(want > 1) and want[-1] > want[0]):
        _no("B should be above 1 and grow with thickness. Check that the lead sweep ran with its six thicknesses.")
        return
    _STATE["p2B"] = True
    _ok("Buildup factors computed. Scattered photons raise the fluence behind the lead above the narrow beam.")
    for xq in (2.0, 4.0):
        i = int(np.argmin(abs(run["x"] - xq)))
        _record("2c. lead x = %g cm: B" % xq, "%.3f +/- %.3f" % (want[i], sB[i]))


def plot_spectrum(run, x=2.0):
    """Plot the photon fluence spectrum just behind the shield."""
    if not isinstance(run, dict):
        _no("Run the lead sweep (2.1) first.")
        return
    i = int(np.argmin(abs(run["x"] - x)))
    sp = run["spectra"][i]
    edges = _spectrum_bins()
    n5 = int(round(660e3 / 5e3))                    # the 5 keV bins from 0 to 660 keV
    cont = sp[:n5] / 5.0                            # per keV
    centers = 0.5 * (edges[:n5] + edges[1:n5 + 1]) / 1e3
    line = sp[-2]
    E90 = LAB["compton"]["E90_keV"]
    tot = run["Phi_total"][i]
    share = sp[:n5][centers < E90].sum() / tot
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.step(centers, np.where(cont > 0, cont, np.nan), where="mid", color=BLUE, lw=1.4,
            label="scattered and secondary photons (per keV)")
    pos = cont[cont > 0]
    lo = pos.min() / 3 if pos.size else 1e-8
    ax.axvline(E0 * 1e3, color=MARI, lw=3, label="662 keV line: narrow-beam fluence %.3g" % line)
    ax.set_yscale("log")
    ax.set_ylim(lo, max(pos.max() * 20 if pos.size else 1, 1e-6))
    ax.set_xlim(0, 700)
    ax.set_xlabel("photon energy just behind the shield (keV)")
    ax.set_ylabel("fluence per keV, per photon per cm2 sent in")
    ax.set_title("662 keV photons after %g cm of %s" % (run["x"][i], run["material"]))
    ax.legend(frameon=False, loc="upper left")
    plt.tight_layout()
    plt.show()
    print("Fluence behind %g cm of %s: total %.4f, in the 662 keV line %.4f, below %d keV %.4f (share of total %.3f)."
          % (run["x"][i], run["material"], tot, line, E90, share * tot, share))
    run["share_below"] = share
    if run["material"] == "lead" and abs(run["x"][i] - 2.0) < 1e-9:
        _record("2d. lead x = 2 cm: share of fluence below %d keV" % E90, "%.3f" % share)
        k = (centers > 70) & (centers < 90)
        _record("2e. lead x = 2 cm: fluence from 70 to 90 keV", "%.2e" % (5.0 * cont[k].sum()))


def narrow_at(run, x):
    """T_narrow at one thickness of a photon sweep."""
    if not isinstance(run, dict):
        _no("Run the lead sweep (2.1) first.")
        return BLANK
    i = int(np.argmin(abs(run["x"] - x)))
    print("x = %g cm: T_narrow = %.5f +/- %.5f" % (run["x"][i], run["T_narrow"][i], run["sT_narrow"][i]))
    return float(run["T_narrow"][i])


def check_mu_rho(mu_rho_lead, mu_rho_water):
    _STATE["p2mr"] = False
    if "mu_lead" not in _STATE or "mu_water" not in _STATE:
        _no("Get PASS on mu for lead and for water first.")
        return
    if _blank([mu_rho_lead, mu_rho_water]):
        _no("A line mu_rho_... = ___ is still blank.")
        return
    ml, mw = _STATE["mu_lead"][0], _STATE["mu_water"][0]
    for name, v, mu, rho in (("lead", mu_rho_lead, ml, RHO["lead"]), ("water", mu_rho_water, mw, RHO["water"])):
        try:
            v = float(v)
        except Exception:
            _no("mu_rho_%s should be one number." % name)
            return
        if abs(v - mu * rho) < 1e-9 * mu * rho and rho != 1:
            _no("mu_rho_%s: divide by the density, do not multiply." % name)
            return
        if abs(v - mu / rho) > 1e-9 * mu / rho:
            _no("mu_rho_%s should be mu_%s / rho_%s." % (name, name, name))
            return
    _STATE["p2mr"] = True
    tl, tw = LAB["s10a"]["muRho"], LAB["s10water"]["muRho"]
    _ok("mu/rho at 662 keV:  lead %.4f cm2/g (Table C.3, p. 595: %.4f);  water %.5f cm2/g (Table C.3, p. 592: %.5f)."
        % (mu_rho_lead, tl, mu_rho_water, tw))
    _ok("Mass per cm2 for one half-thickness:  lead %.2f g/cm2;  water %.2f g/cm2."
        % (math.log(2) / mu_rho_lead, math.log(2) / mu_rho_water))
    _record("2f. mu/rho lead, water (cm2/g)", "%.4f, %.5f" % (mu_rho_lead, mu_rho_water))
    xw = math.log(2) / mw
    _record("2g. x_half, water (cm)", "%.2f (S10: %.2f)" % (xw, LAB["s10water"]["xHalf"]))


def compare_buildup(lead, water):
    """Plot the buildup factor against the number of mean free paths, mu x, for lead and water."""
    if not (isinstance(lead, dict) and isinstance(water, dict)) or "mu_lead" not in _STATE or "mu_water" not in _STATE or "B" not in lead:
        _no("Get PASS on mu for both materials and on B first.")
        return
    Bw = water["Phi_total"] / water["Phi_narrow"]
    sBw = Bw * np.sqrt((water["sPhi_total"] / water["Phi_total"]) ** 2 + (water["sPhi_narrow"] / water["Phi_narrow"]) ** 2)
    water["B"], water["sB"] = Bw, sBw
    ml, mw = _STATE["mu_lead"][0], _STATE["mu_water"][0]
    fig, ax = plt.subplots()
    ax.errorbar(ml * lead["x"], lead["B"], yerr=lead["sB"], fmt="o-", color=INK, ms=5, capsize=3, label="lead")
    ax.errorbar(mw * water["x"], Bw, yerr=sBw, fmt="s-", color=BLUE, ms=5, capsize=3, label="water")
    ax.set_xlabel(r"shield thickness in mean free paths, $\mu x$")
    ax.set_ylabel("fluence buildup factor B")
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()
    print("  water x (cm)   mu x    B")
    for x, b, s in zip(water["x"], Bw, sBw):
        print("  %8.1f      %5.2f   %.2f +/- %.2f" % (x, mw * x, b, s))


# ------------------------------------------------------------------ Part 3
def _uGyh(D_per_phi0):
    return D_per_phi0 * LAB["src"]["phi0"] * 3600 * 1e6


def check_D_narrow(D_narrow, T_narrow_2cm):
    _STATE["p3a"] = False
    if _blank(D_narrow):
        _no("The line D_narrow = ___ is still blank.")
        return
    K, mu_en, phi0 = LAB["s12a"]["K"], LAB["s12a"]["muenW"], LAB["src"]["phi0"]
    want = K * E0 * mu_en * phi0 * T_narrow_2cm
    try:
        dv = float(D_narrow)
    except Exception:
        _no("D_narrow should be one number, in Gy per second.")
        return
    if abs(dv - want) <= 1e-6 * want:
        _STATE["p3a"] = True
        _STATE["D_narrow"] = dv
        _ok("Narrow-beam dose rate behind 2 cm of lead: %.3e Gy/s = %.0f microGy/h." % (dv, dv * 3.6e9))
        _record("3a. D_narrow behind 2 cm lead (microGy/h)", "%.0f" % (dv * 3.6e9))
        return
    cases = [(K * E0 * LAB["s10water"]["muRho"] * phi0 * T_narrow_2cm,
              "You used mu/rho. Dose counts the energy the water absorbs, so Eq. 9.6 uses mu_en/rho (S12-A)."),
             (K * E0 * mu_en * phi0, "This is the dose rate with no shield. Multiply by T_narrow_2cm."),
             (1.602e-13 * E0 * mu_en * phi0 * T_narrow_2cm, "Units: 1.602e-10 converts MeV per gram to J per kg. Use 1.602e-10."),
             (K * E0 * 1e3 * mu_en * phi0 * T_narrow_2cm, "E0 is in MeV (0.662), not keV.")]
    for v, msg in cases:
        if abs(dv - v) <= 1e-6 * abs(v):
            _no(msg)
            return
    _no("D_narrow should be 1.602e-10 * E0 * muen_rho * phi0 * T_narrow_2cm (Eq. 9.6 in rate form, as in S12-A).")


def dose_behind(material, x, n_particles=400_000):
    """Run OpenMC for one shield and return the dose rate behind it, from the full photon spectrum."""
    if _blank(x):
        _no("The thickness is still blank.")
        return BLANK
    if not _ready("photon"):
        return BLANK
    part = 3 if material == "lead" else 4
    model, wd = _model(material, float(x), "photon", n_particles, _seed(part), True)
    out, dt = _run(model, wd)
    phi_tot = out["fluence"][0][0] / DELTA
    phi_n, sphi_n = out["fluence narrow"][0][0] / DELTA, out["fluence narrow"][1][0] / DELTA
    D, sD = out["dose"][0][0] / DELTA, out["dose"][1][0] / DELTA
    T_n = out["narrow"][0][0]
    r = dict(material=material, x=float(x), T_narrow=T_n, Phi_total=phi_tot, Phi_narrow=phi_n,
             B_flux=phi_tot / phi_n, D_total=D * LAB["src"]["phi0"], sD_total=sD * LAB["src"]["phi0"], seconds=dt)
    print("%s, x = %.2f cm, %s photons, %.1f s of OpenMC time" % (material, x, f"{n_particles:,}", dt))
    print("  narrow-beam transmission T_narrow = %.5f" % T_n)
    print("  fluence buildup B = Phi_total / Phi_narrow = %.3f" % r["B_flux"])
    print("  dose rate from the full spectrum, D_total = %.3e +/- %.1e Gy/s = %.0f microGy/h" % (
        r["D_total"], r["sD_total"], r["D_total"] * 3.6e9))
    return r


def check_B_dose(B_dose, dose):
    _STATE["p3b"] = False
    if dose is BLANK or not isinstance(dose, dict):
        _no("Run the dose_behind cell first.")
        return
    if "D_narrow" not in _STATE:
        _no("Get PASS on D_narrow first.")
        return
    if _blank(B_dose):
        _no("The line B_dose = ___ is still blank.")
        return
    want = dose["D_total"] / _STATE["D_narrow"]
    try:
        bv = float(B_dose)
    except Exception:
        _no("B_dose should be one number.")
        return
    if abs(bv - 1 / want) < 1e-9 * (1 / want):
        _no("B_dose is upside down: total dose / narrow-beam dose.")
        return
    if abs(bv - want) > 1e-9 * want:
        _no("B_dose should be D_total / D_narrow, with D_total = dose_lead[\"D_total\"].")
        return
    sB = want * dose["sD_total"] / dose["D_total"]
    if not (1 < want < dose["B_flux"]):
        _no("B_dose = %.3f should sit between 1 and the fluence buildup %.3f. Check that dose_behind ran for lead at 2 cm." % (want, dose["B_flux"]))
        return
    _STATE["p3b"] = True
    _STATE["B_dose_lead"] = want
    _ok("Dose buildup behind 2 cm of lead: B_dose = %.3f +/- %.3f, below the fluence buildup B = %.3f."
        % (want, sB, dose["B_flux"]))
    _record("3b. lead 2 cm: D_total (microGy/h)", "%.0f +/- %.0f" % (dose["D_total"] * 3.6e9, dose["sD_total"] * 3.6e9))
    _record("3b. lead 2 cm: B_dose, B_fluence", "%.3f +/- %.3f, %.3f" % (want, sB, dose["B_flux"]))


def check_x_water(x_water, mu_lead, mu_water):
    _STATE["p3c"] = False
    if _blank(x_water):
        _no("The line x_water = ___ is still blank.")
        return
    want = mu_lead * 2.0 / mu_water
    try:
        xv = float(x_water)
    except Exception:
        _no("x_water should be one number.")
        return
    if abs(xv - mu_water * 2.0 / mu_lead) < 1e-9:
        _no("x_water is upside down. Water attenuates less per cm, so the water shield must be thicker than 2 cm.")
        return
    if abs(xv - want) > 1e-9 * want:
        _no("Same narrow-beam attenuation means the same mu x: mu_water * x_water = mu_lead * 2.0. Solve for x_water.")
        return
    _STATE["p3c"] = True
    _ok("Water with the same narrow-beam attenuation as 2 cm of lead: %.2f cm (%.1f g/cm2 against %.1f g/cm2 of lead)."
        % (want, want * RHO["water"], 2.0 * RHO["lead"]))
    _record("3c. x_water (cm)", "%.2f" % want)


def compare_dose(dose_lead, dose_water):
    if not isinstance(dose_water, dict) or not isinstance(dose_lead, dict):
        _no("Run both dose cells first.")
        return
    Dn = _STATE.get("D_narrow")
    print("                       lead, %.2f cm      water, %.2f cm" % (dose_lead["x"], dose_water["x"]))
    print("  T_narrow             %.5f            %.5f" % (dose_lead["T_narrow"], dose_water["T_narrow"]))
    print("  D_total (microGy/h)  %.0f                %.0f" % (dose_lead["D_total"] * 3.6e9, dose_water["D_total"] * 3.6e9))
    print("  fluence buildup B    %.2f               %.2f" % (dose_lead["B_flux"], dose_water["B_flux"]))
    if Dn:
        K, mu_en, phi0 = LAB["s12a"]["K"], LAB["s12a"]["muenW"], LAB["src"]["phi0"]
        Bw = dose_water["D_total"] / (K * E0 * mu_en * phi0 * dose_water["T_narrow"])
        print("  dose buildup B_dose  %.2f               %.2f" % (dose_lead["D_total"] / Dn, Bw))
        _record("3d. water %.1f cm: D_total (microGy/h), B_dose" % dose_water["x"],
                "%.0f, %.2f" % (dose_water["D_total"] * 3.6e9, Bw))
    print("\nS12-A, from the uncollided flux behind 2 cm of lead plus 10 cm of concrete: %d microGy/h." % LAB["s12a"]["DuGyh"])


# ------------------------------------------------------------------ 665
def try_build(build):
    """[665] Call your build_model function; report a blank or an error without stopping the notebook."""
    import traceback
    try:
        model = build()
    except Exception as ex:
        tb = traceback.extract_tb(ex.__traceback__)
        line = [f for f in tb if f.name == build.__name__]
        where = " (line %d of the cell: %s)" % (line[-1].lineno, line[-1].line) if line else ""
        if "still blank" in str(ex) or "_Blank" in str(ex) or "BLANK" in str(ex):
            _no("A line in build_model is still blank%s." % where)
        else:
            _no("build_model stopped%s: %s" % (where, str(ex).strip().splitlines()[-1] if str(ex).strip() else type(ex).__name__))
        return BLANK
    if not isinstance(model, openmc.Model):
        _no("build_model should return the openmc.Model it builds.")
        return BLANK
    return model


def run_665(model):
    """[665] Run your model and read your tallies."""
    if model is BLANK:
        _no("Build the model first.")
        return BLANK
    model.settings.seed = _seed(6)
    model.settings.output = {"tallies": False}
    wd = tempfile.mkdtemp(prefix="labB665_")
    try:
        out, dt = _run(model, wd)
    except Exception as ex:
        msg = str(ex).strip().splitlines()[-1] if str(ex).strip() else repr(ex)
        _no("OpenMC stopped: %s\n          A blank (___) in the model, or a missing photon_transport = True, causes this." % msg)
        return BLANK
    print("Your model ran in %.1f s." % dt)
    return out


def check_665(out, lead):
    if out is BLANK or not isinstance(out, dict):
        _no("Run your model first.")
        return
    if not isinstance(lead, dict) or "mu_lead" not in _STATE:
        _no("Finish Part 2 (the lead sweep and mu) first; this check compares your model with it.")
        return
    need = ["narrow", "strict", "mesh total", "mesh narrow"]
    miss = [n for n in need if n not in out]
    if miss:
        _no("Tallies missing from your model: %s. Keep the tally names as written." % ", ".join(miss))
        return
    T, sT = float(out["narrow"][0][0]), float(out["narrow"][1][0])
    i = int(np.argmin(abs(lead["x"] - 2.0)))
    Tr, sTr = lead["T_narrow"][i], lead["sT_narrow"][i]
    z = (T - Tr) / math.sqrt(sT ** 2 + sTr ** 2)
    if T <= 0:
        _no("Your narrow-beam tally is zero. Check the source energy (eV) and the energy filter around it.")
        return
    if abs(z) > 4:
        _no("Your T_narrow = %.5f +/- %.5f differs from the helper's %.5f +/- %.5f by %.1f sigma. "
            "Check the density line and the back-face position." % (T, sT, Tr, sTr, z))
        return
    Ts, sTs = float(out["strict"][0][0]), float(out["strict"][1][0])
    mu_s = -math.log(Ts) / 2.0
    _ok("Your model: T_narrow = %.5f +/- %.5f; the helper's lead run gives %.5f +/- %.5f (%+.2f sigma)." % (T, sT, Tr, sTr, z))
    _ok("Strictly uncollided (no change of direction): T = %.5f +/- %.5f, so mu = %.4f per cm and x_half = %.4f cm."
        % (Ts, sTs, mu_s, math.log(2) / mu_s))
    _record("665a. your T_narrow, 2 cm lead", "%.5f +/- %.5f (%+.2f sigma)" % (T, sT, z))
    _record("665b. strictly uncollided T, mu, x_half", "%.5f, %.4f per cm, %.4f cm" % (Ts, mu_s, math.log(2) / mu_s))
    mt, mn = out["mesh total"][0], out["mesh narrow"][0]
    nb = mt.size
    dx = 2.0 / nb
    ft, fn = mt / dx, mn / dx
    zc = (np.arange(nb) + 0.5) * dx
    fig, ax = plt.subplots()
    ax.semilogy(zc, ft, "o-", color=MARI, ms=4, label="total photon fluence")
    ax.semilogy(zc, fn, "s-", color=BLUE, ms=4, label="662 keV (narrow-beam) fluence")
    ax.semilogy(zc, np.exp(-_STATE["mu_lead"][0] * zc), color=INK, lw=1, ls="--", label=r"$e^{-\mu x}$ with your Part 2 $\mu$")
    ax.set_xlabel("depth in the lead (cm)")
    ax.set_ylabel("fluence per photon per cm2 sent in")
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()
    ratio = ft / fn
    print("  depth (cm)   total     narrow    total / narrow")
    for k in range(0, nb, max(1, nb // 10)):
        print("  %6.2f      %.4f    %.4f    %.3f" % (zc[k], ft[k], fn[k], ratio[k]))
    print("  %6.2f      %.4f    %.4f    %.3f" % (zc[-1], ft[-1], fn[-1], ratio[-1]))
    _record("665c. mesh: total fluence, first and last bin", "%.3f, %.4f" % (ft[0], ft[-1]))
    _record("665c. mesh: total / narrow, first and last bin", "%.3f, %.3f" % (ratio[0], ratio[-1]))


def answer_sheet():
    if not ANSWERS:
        _no("No results yet. Run the parts above first.")
        return
    if not _STATE.get("setup"):
        print("WARNING: setup did not PASS (Part 0), so these values may not be your own. "
              "Set MY_ID, then Runtime > Run all.\n")
    print("Copy these values to your answer sheet.\n")
    for k in sorted(ANSWERS, key=lambda s: (s.split(".")[0].zfill(4), _ORDER[s])):
        print("  %-54s %s" % (k, ANSWERS[k]))


print("Helper code loaded. Next: run check_setup() in the cell below.")


**0.4 Setup check.** Run the next cell. The setup check must print **PASS**. The check runs OpenMC once: 20,000 neutrons through 1 cm of water with scattering switched off, compared with $e^{-\Sigma_t x}$.

**Checkpoint:** take a screenshot of the PASS line and submit it on WebCampus. *Answer sheet 0a and 0b.*

In [ ]:
check_setup()

---
## Part 1 · Same slab, production code

Lab A followed neutrons through water at one speed, with isotropic scattering and the cross sections from S9-A: $\Sigma_t = 2.1963$, $\Sigma_s = 2.174$ and $\Sigma_a = 0.0223$ cm⁻¹. OpenMC can run exactly that simplified physics in its **multigroup mode**: you give it the cross sections for one energy group, and it needs no nuclear data file. The geometry is Lab A's: neutrons hit the front face at normal incidence, and a neutron that leaves through either face never returns.

OpenMC gives every result with its σ. OpenMC splits the neutrons into 20 batches and uses the spread of the 20 batch results. Lab A used the counting formula $\sigma_T = \sqrt{T(1-T)/N}$; both methods estimate the same uncertainty.

**1.1 Pure absorber.** With scattering switched off, every collision removes the neutron, so only uncollided neutrons get through. Four slabs, 100,000 neutrons each.

In [ ]:
Sigma_t = WATER["Sigma_t"]      # 2.1963 per cm, water, thermal neutron (S9-A)
res0 = run_sweep("water", [0.5, 1, 2, 3], particle="neutron", n_particles=100_000, scattering=False, part=1)

**1.2 Your line.** The uncollided transmission $e^{-\Sigma_t x}$ (S&F Eq. 7.6, p. 181) at every thickness. `x` holds the four thicknesses, so one line gives all four values. Numpy writes $e^{(\dots)}$ as `np.exp(...)`.

In [ ]:
x = res0["x"]                          # the four thicknesses, cm
T_exact = ___       # uncollided transmission at each thickness
check_part1a(T_exact, res0)

*Answer sheet 1a.*

**1.3 With scattering.** The same eight slabs as Lab A Part 4, from 0.5 to 10 cm, with 200,000 neutrons each: ten times Lab A's 20,000. About 15 seconds.

In [ ]:
res1 = run_sweep("water", [0.5, 1, 2, 3, 4, 6, 8, 10], particle="neutron", n_particles=200_000, part=2)

**1.4 Your Lab A result.** Copy $T_{total}$ at 2 cm and its σ from your Lab A answer sheet, item 4a. No Lab A result? Use the values in the comments.

In [ ]:
T_labA = ___          # your Lab A T_total at 2 cm (answer sheet 4a). No Lab A result: use 0.2559
sigma_labA = ___      # its sigma (answer sheet 4a). No Lab A result: use 0.0031

**1.5 Your line.** How far apart are OpenMC's result and yours, in units of their combined uncertainty? For two independent results, the uncertainties add in quadrature:

$$z = \frac{T_{OpenMC} - T_{Lab\,A}}{\sqrt{\sigma_{OpenMC}^2 + \sigma_{Lab\,A}^2}}$$

A $|z|$ below 2 is ordinary scatter. A $|z|$ above 3 means the two calculations differ.

In [ ]:
T_openmc, sigma_openmc = at(res1, 2.0)
z = ___
check_part1b(z, T_openmc, sigma_openmc, T_labA, sigma_labA, res1)

*Answer sheet 1b and 1c.*

**1.6 Under the hood.** The next cell prints the four input files OpenMC read for the last run, the 10 cm slab: materials, geometry, settings and tallies (a tally is a quantity OpenMC adds up over all the particles). Engineers write models like these, or the Python that writes them. *Answer sheet 1d* asks you to find three things in the printout.

In [ ]:
show_model()

---
## Part 2 · Cs-137 photons through lead and water

In Part 2 OpenMC runs in **continuous-energy mode**. OpenMC reads evaluated photon cross sections for each element from data files (ENDF/B-VII.1; its photon data come from the EPDL library, the same family of data as S&F Fig. 7.3, p. 194). OpenMC follows every photoelectric absorption, Compton scattering, coherent scattering and pair production event, and the x-rays and electrons these events produce.

Each photon starts at 662 keV, the Cs-137 line from S10-A, moving straight into the slab. For each thickness the helper reports three quantities:

- **T_narrow:** the share of photons that leave the back face still at 662 keV. These photons never lost energy, so they are the narrow beam that $e^{-\mu x}$ describes.
- **Phi_total:** the fluence of all photons just behind the shield, scattered photons included, per photon per cm² sent in. **Phi_narrow** is the same quantity for the 662 keV photons only.
- the energy spectrum of the photons just behind the shield.

S10 (§7.1.5, p. 183) said that $e^{-\mu x}$ counts only photons that never interacted, so it is a floor. A buildup factor B adds the scattered photons back, $\Phi_{total} = B\,\Phi_{uncollided}$, and B "comes from transport codes or Monte Carlo". Part 2 computes B with OpenMC.

**2.1 Lead.** Six thicknesses from 0.5 to 6 cm, 200,000 photons each. About 30 seconds.

In [ ]:
lead = run_sweep("lead", [0.5, 1, 2, 3, 4, 6], particle="photon", energy_MeV=0.662, n_particles=200_000, part=21)

**2.2 Your line: the narrow-beam μ.** Rearrange $T_{narrow} = e^{-\mu x}$ (S&F Eq. 7.6, p. 181) for μ, the linear attenuation coefficient. One line gives one μ per thickness; the check combines them into one weighted mean, `mu_lead`.

In [ ]:
T_narrow, x = lead["T_narrow"], lead["x"]
mu = ___            # linear attenuation coefficient from each thickness, per cm
mu_lead = check_mu(lead, mu)

**2.3 Your line: the half-thickness.** $x_{1/2} = \ln 2 / \mu$ (S&F Eq. 7.9, p. 182). S10-A found 0.585 cm from Table C.3.

In [ ]:
x_half = ___          # half-thickness of lead at 662 keV, cm
check_xhalf(x_half, mu_lead)

The check compares your half-thickness with the data library OpenMC reads, not with Table C.3. The two data sets differ by about 1%, and 200,000 photons per slab are enough to resolve a 1% difference. *Answer sheet 2a and 2b.*

**2.4 Your line: the buildup factor.** $B = \Phi_{total} / \Phi_{narrow}$ at each thickness.

In [ ]:
Phi_total, Phi_narrow = lead["Phi_total"], lead["Phi_narrow"]
B = ___            # fluence buildup factor at each thickness
check_B(B, lead)

*Answer sheet 2c.*

**2.5 The spectrum behind 2 cm of lead.** Read the plot from right to left: the 662 keV line, then the Compton-scattered photons below it, then two small peaks between 70 and 90 keV. S10's Compton slide gives the energies a 662 keV photon keeps after one scattering: 288 keV at 90° and 184 keV straight back (S&F Eq. 7.33, p. 193). *Answer sheet 2d and 2e.*

In [ ]:
plot_spectrum(lead, 2.0)

**2.6 Water.** The same 662 keV photons through water. Water attenuates far less per centimetre, so the slabs run from 5 to 30 cm. About 20 seconds.

In [ ]:
water = run_sweep("water", [5, 10, 20, 30], particle="photon", energy_MeV=0.662, n_particles=100_000, part=22)

**2.7 Your line.** μ for water: the same line as in 2.2.

In [ ]:
T_narrow, x = water["T_narrow"], water["x"]
mu = ___            # the same line as in 2.2
mu_water = check_mu(water, mu)

**2.8 Your lines: the mass attenuation coefficient** $\mu/\rho$ (S&F Eq. 7.11, p. 184). S10 showed that $\mu/\rho$ depends on the material and the photon energy, not on the density.

In [ ]:
rho_lead, rho_water = RHO["lead"], RHO["water"]     # 11.35 and 1.00 g/cm3
mu_rho_lead = ___                  # cm2/g
mu_rho_water = ___               # cm2/g
check_mu_rho(mu_rho_lead, mu_rho_water)

*Answer sheet 2f.*

**2.9 Buildup in lead and in water.** The plot puts both materials on one axis: the shield thickness in mean free paths, $\mu x$. At the same $\mu x$ the narrow beam is the same in both materials, so any difference in B comes from the scattered photons. *Answer sheet 2g.*

In [ ]:
compare_buildup(lead, water)

---
## Part 3 · Dose behind the shield

S10's shielding problem has a 10 Ci Cs-137 source 2 m from the shield. The source emits $S = 3.141 \times 10^{11}$ photons per second, so the flux at the shield is $\phi_0 = S/(4\pi r^2) = 6.249 \times 10^{5}$ photons per cm² per second. S12-A took S10's uncollided flux behind 2 cm of lead and 10 cm of concrete and found 121 μGy/h with S&F Eq. 9.6 (p. 275) in rate form:

$$\dot D = 1.602\times10^{-10}\; E\; \frac{\mu_{en}}{\rho}\; \phi$$

with $\dot D$ in Gy/s, $E$ in MeV, $\mu_{en}/\rho$ in cm²/g and $\phi$ in photons per cm² per second.

S12-A's dose used the uncollided flux only: the floor. Here the shield is the 2 cm of lead alone, and OpenMC adds the scattered photons. Each photon's dose uses $\mu_{en}/\rho$ for water at that photon's own energy (Table C.3, p. 592). Two metres from the source, the photons that reach a 2 cm shield travel almost parallel, as in the slab runs. S&F notes (p. 183) that a buildup factor is most often applied to dose rather than to the number of photons; Part 3 shows why the difference matters.

In [ ]:
E0 = LAB["E_MeV"]                  # 0.662 MeV, Cs-137
muen_rho = LAB["s12a"]["muenW"]    # 0.03257 cm2/g, water mu_en/rho at 662 keV (S12-A; Table C.3, p. 592)
phi0 = LAB["src"]["phi0"]          # photons per cm2 per s at the shield (S10)
T_narrow_2cm = narrow_at(lead, 2.0)

**3.1 Your line: the narrow-beam dose rate.** Eq. 9.6 in rate form, with the narrow-beam flux behind 2 cm of lead, $\phi_0\,T_{narrow}$. The result is in Gy/s.

In [ ]:
D_narrow = ___     # Gy/s
check_D_narrow(D_narrow, T_narrow_2cm)

*Answer sheet 3a.*

**3.2 The full spectrum.** OpenMC sends 400,000 photons through 2 cm of lead and adds up the dose from every photon that comes out, whatever its energy.

In [ ]:
dose_lead = dose_behind("lead", 2.0)

**3.3 Your line: the dose buildup factor.** $B_{dose} = \dot D_{total} / \dot D_{narrow}$.

In [ ]:
D_total = dose_lead["D_total"]          # Gy/s, from the full spectrum
B_dose = ___
check_B_dose(B_dose, dose_lead)

*Answer sheet 3b.*

**3.4 Your line: a water shield.** How thick must a water shield be to give the same narrow-beam attenuation as 2 cm of lead? The narrow beam is the same when $\mu x$ is the same.

In [ ]:
x_water = ___     # cm of water with the same mu x as 2 cm of lead
check_x_water(x_water, mu_lead, mu_water)

**3.5 Dose behind the water shield.** The same calculation as 3.2, for the water shield, and a table comparing the two shields. *Answer sheet 3c to 3e.*

In [ ]:
dose_water = dose_behind("water", x_water)
compare_dose(dose_lead, dose_water)

---
## [665] Build the lead model yourself

The function below builds the 2 cm lead model of Part 2 with the OpenMC Python API (documentation: docs.openmc.org). Fill in the blanks. Keep the tally names as written; the check reads them.

Two tallies are new:

- **strict:** photons that cross the back face at 662 keV **and** within 0.08° of straight through (the cosine of the angle to the surface normal above 0.999999). These photons never interacted at all.
- **mesh total** and **mesh narrow:** the photon fluence in 20 thin layers through the lead, for all photons and for 662 keV photons only.

In [ ]:
def build_model():
    x_pb = 2.0                                               # cm
    lead_mat = openmc.Material(name="lead")
    lead_mat.add_nuclide("Pb208", 1.0)                       # one isotope stands in for natural lead (photon physics depends on the element)
    N_lead = ___   # atoms per barn-cm: rho N_a / A (S&F Eq. 7.10), with 1 barn = 1e-24 cm2
    lead_mat.set_density("atom/b-cm", N_lead)
    mats = openmc.Materials([lead_mat])
    mats.cross_sections = XS

    front = openmc.XPlane(0.0, boundary_type="vacuum")
    back = openmc.XPlane(___, boundary_type="vacuum")
    sides = (+openmc.YPlane(-1, boundary_type="reflective") & -openmc.YPlane(1, boundary_type="reflective")
             & +openmc.ZPlane(-1, boundary_type="reflective") & -openmc.ZPlane(1, boundary_type="reflective"))
    shield = openmc.Cell(fill=lead_mat, region=___ & sides)   # between the two faces
    geometry = openmc.Geometry([shield])

    settings = openmc.Settings()
    settings.run_mode = "fixed source"
    settings.batches, settings.particles = 20, 10_000
    settings.photon_transport = ___
    settings.source = openmc.IndependentSource(
        space=openmc.stats.Point((1e-9, 0.0, 0.0)),
        angle=openmc.stats.Monodirectional((1.0, 0.0, 0.0)),
        energy=openmc.stats.Discrete([___], [1.0]),       # source energy in eV
        particle=___)

    photons = openmc.ParticleFilter(["photon"])
    line = openmc.EnergyFilter([0.662e6 - 1, 0.662e6 + 1])      # the 662 keV line
    narrow = openmc.Tally(name="narrow")
    narrow.filters = [openmc.SurfaceFilter(back), photons, line]
    narrow.scores = [___]                                # photons crossing the back face
    strict = openmc.Tally(name="strict")
    strict.filters = [openmc.SurfaceFilter(back), photons, line, openmc.MuSurfaceFilter([___, 1.0])]
    strict.scores = ["current"]
    mesh = openmc.RegularMesh()
    mesh.lower_left = (0.0, -1.0, -1.0)
    mesh.upper_right = (___, 1.0, 1.0)
    mesh.dimension = (___, 1, 1)                                # 20 layers through the lead
    mesh_total = openmc.Tally(name="mesh total")
    mesh_total.filters = [openmc.MeshFilter(mesh), photons]
    mesh_total.scores = [___]
    mesh_narrow = openmc.Tally(name="mesh narrow")
    mesh_narrow.filters = [openmc.MeshFilter(mesh), photons, line]
    mesh_narrow.scores = ["flux"]
    return openmc.Model(geometry, mats, settings, openmc.Tallies([narrow, strict, mesh_total, mesh_narrow]))


model = try_build(build_model)     # [665]
out665 = run_665(model)
check_665(out665, lead)

*Answer sheet 665a to 665c.*

---
## Answer-sheet values

Run the cell below and copy every value to the answer sheet. Then download this notebook (*File > Download > Download .ipynb*) and submit both files on WebCampus.

In [ ]:
answer_sheet()